In [0]:
%sql
CREATE OR REPLACE TABLE data_warehouse_factory.gold.dim_employee_assignments AS
WITH deduplicated AS (
  SELECT 
    upper(trim(line_code)) AS line_code,
    trim(line_name) AS line_name,
    upper(trim(cell_code)) AS cell_code,
    trim(cell_name) AS cell_name,
    CAST(default_employee_key AS INT) AS default_employee_key,
    trim(default_employee_name) AS default_employee_name,
    CAST(backup_employee_key AS INT) AS backup_employee_key,
    trim(backup_employee_name) AS backup_employee_name,
    upper(trim(coalesce(shift, 'ALL SHIFTS'))) AS shift,
    CAST(valid_from AS DATE) AS valid_from,
    remarks,
    source,
    _silver_ingested_at,
    ROW_NUMBER() OVER (
      PARTITION BY upper(trim(line_code)), upper(trim(cell_code)), upper(trim(coalesce(shift, 'ALL SHIFTS'))), CAST(valid_from AS DATE)
      ORDER BY _silver_ingested_at DESC
    ) AS rn
  FROM data_warehouse_factory.silver.silver_employee_assignments
  WHERE line_code IS NOT NULL AND cell_code IS NOT NULL
),
scd2_prep AS (
  SELECT 
    *,
    LEAD(valid_from) OVER (
      PARTITION BY line_code, cell_code, shift 
      ORDER BY valid_from ASC
    ) AS next_valid_from
  FROM deduplicated
  WHERE rn = 1
)
SELECT 
  md5(concat_ws('||', line_code, cell_code, shift, cast(valid_from as string))) AS assignment_key,
  md5(concat_ws('||', line_code, cell_code)) AS cell_key,
  line_code,
  line_name,
  cell_code,
  cell_name,
  default_employee_key,
  default_employee_name,
  backup_employee_key,
  backup_employee_name,
  shift,
  valid_from,
  COALESCE(DATE_ADD(next_valid_from, -1), DATE '9999-12-31') AS valid_to,
  CASE WHEN next_valid_from IS NULL THEN TRUE ELSE FALSE END AS is_current,
  remarks,
  source,
  CURRENT_TIMESTAMP() AS _gold_created_at
FROM scd2_prep;